# Contextual RDM simulation

This example supplies the Racing Diffusion Model's per-trial `correct_idx` input through a `ContextSimulator`. `ContextMapping` routes that generated variable to the simulator.

In [1]:
import numpy as np
import superstats as sup

INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/lschumacher/.local/share/uv/python/cpython-3.13.14-macos-aarch64-none/lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
INFO:bayesflow:Using backend 'jax'


In [2]:
def generate_context(batch_size, num_steps):
    correct_idx = np.tile(np.arange(num_steps) % 2, (batch_size, 1)).astype(np.int32)
    return {"correct_idx": correct_idx}

context = sup.ContextSimulator(generate_context, is_batched=True)
context_mapping = sup.ContextMapping(simulator_context=("correct_idx",))

In [ ]:
prior = sup.JointPrior(
    v_base=sup.Prior("logistic", loc=0.0, scale=1.0),
    v_diff=sup.transition.RandomWalk(),
    a_base=1.0,
    tau=0.2,
    bias=1.0,
    sigma_diff=1.0,
)

model = sup.Model(
    prior=prior,
    simulator=sup.simulation.sample_rdm,
    missing=None,
    context=context,
    context_mapping=context_mapping,
)

In [4]:
batch_size, num_steps = 3, 6
samples = model.sample(batch_size=batch_size, num_steps=num_steps)
print(samples["response_time"].shape, samples["choice"].shape, samples["correct_idx"].shape)

(3, 6) (3, 6) (3, 6)


`correct_idx` matches a named RDM simulator argument, so `Model` supplies it directly rather than requiring `sample_rdm` to accept a dictionary. Context variables that do not match simulator arguments are forwarded as `context=` to simulators that declare that keyword.